# Model Router Evaluation — Interactive Walkthrough

Step through the evaluation pipeline interactively. **No API keys needed** — this notebook uses mock data to show every metric, chart, and output the tool produces.

| What you'll see | Section |
|----------------|---------|
| Sample dataset schema | §1 Setup |
| Cost & latency metrics | §4 Metrics |
| Quality (judge) metrics | §5 Quality |
| All 8 chart types inline | §6 Charts |
| Full HTML dashboard | §7 Dashboard |
| Go-live checklist | §9 Go Live |
| **Foundry cloud eval (optional)** | **§10 Foundry** |
| **Cross-validate local vs Foundry** | **§11 Cross-Validate** |

## 1. Setup

Import the evaluation library. Run this once — everything else builds on it.

In [ ]:
import sys, random
from pathlib import Path

# Project root = directory containing this notebook (repo root)
project_root = Path.cwd()
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

from src.client import CompletionResult
from src.config import EvalConfig, EndpointConfig, PricingConfig
from src.dataset import Prompt, load_dataset
from src.judge import AbsoluteScore, JudgeResult
from src.metrics import compute_metrics, compute_quality_metrics

print(f"Project root: {project_root}")
print("All imports OK ✓")

## 2. Load & Inspect the Dataset

The sample dataset ships with the repo. Each prompt has an `id`, `prompt` text, and optional `category`/`difficulty`.

`load_dataset()` supports **JSONL**, **CSV**, and **SQL databases** (SQLite built-in, others via SQLAlchemy). The file format is auto-detected by extension (`.jsonl`, `.csv`) or connection string (`sqlite:///...`).

In [ ]:
dataset_path = project_root / "datasets" / "sample_custom.jsonl"
prompts_real = load_dataset(dataset_path)

print(f"Loaded {len(prompts_real)} prompts from {dataset_path.name}\n")
print(f"{'ID':<8} {'Category':<24} {'Difficulty':<10} Prompt (first 60 chars)")
print("-" * 100)
for p in prompts_real[:5]:
    print(f"{p.id:<8} {(p.category or '-'):<24} {(p.difficulty or '-'):<10} {p.prompt[:60]}...")

if len(prompts_real) > 5:
    print(f"  ... and {len(prompts_real) - 5} more")

## 3. Generate Mock Evaluation Results

In a real run, `runner.py` calls the Model Router and baseline endpoints. Here we generate **100 synthetic results** with realistic latency and token distributions.

In [ ]:
# -- Configuration (same as scripts/generate_sample_report.py) --
NUM_PROMPTS = 100
CATEGORIES = [
    "code_generation", "technical_knowledge", "creative_writing", "math",
    "reasoning", "summarization", "general_knowledge", "instruction_following",
]

LATENCY_PROFILES = {
    "model_router": {
        "code_generation": (320, 80), "technical_knowledge": (280, 60),
        "creative_writing": (350, 90), "math": (250, 50),
        "reasoning": (380, 100), "summarization": (220, 40),
        "general_knowledge": (260, 55), "instruction_following": (300, 70),
    },
    "baseline": {
        "code_generation": (450, 90), "technical_knowledge": (420, 70),
        "creative_writing": (480, 100), "math": (380, 60),
        "reasoning": (520, 120), "summarization": (350, 50),
        "general_knowledge": (400, 65), "instruction_following": (430, 80),
    },
}

TOKEN_PROFILES = {
    "model_router": (80, 30, 200, 80),
    "baseline": (80, 30, 250, 100),
}


def make_prompts(n, seed=42):
    rng = random.Random(seed)
    return [
        Prompt(
            id=f"mock-{i+1:04d}",
            prompt=f"[Mock prompt {i+1}] Category: {CATEGORIES[i % len(CATEGORIES)]}",
            category=CATEGORIES[i % len(CATEGORIES)],
            difficulty=rng.choice(["easy", "medium", "hard"]),
        )
        for i in range(n)
    ]


def make_results(prompts, endpoint, seed=42, error_rate=0.02):
    rng = random.Random(seed)
    key = "model_router" if "router" in endpoint else "baseline"
    tp = TOKEN_PROFILES[key]
    lp = LATENCY_PROFILES[key]
    results = []
    for p in prompts:
        is_err = rng.random() < error_rate
        cat = p.category or "general_knowledge"
        lat_m, lat_s = lp.get(cat, (350, 80))
        latency = max(50, rng.gauss(lat_m, lat_s))
        pt = max(10, int(rng.gauss(tp[0], tp[1])))
        ct = max(10, int(rng.gauss(tp[2], tp[3])))
        results.append(CompletionResult(
            request_id=f"req-{endpoint}-{p.id}", prompt_id=p.id, endpoint=endpoint,
            model_name="gpt-4o-mini" if "router" in endpoint else "gpt-4o",
            response_text="" if is_err else f"[Mock response for {p.id}]",
            prompt_tokens=0 if is_err else pt, completion_tokens=0 if is_err else ct,
            total_tokens=0 if is_err else pt + ct,
            latency_ms=round(latency, 2), status="error" if is_err else "success",
            error_message="Mock error" if is_err else None, timestamp="2026-04-22T12:00:00Z",
        ))
    return results


# -- Generate --
prompts = make_prompts(NUM_PROMPTS)
router_results = make_results(prompts, "model_router", seed=42)
baseline_results = make_results(prompts, "baseline:gpt-4o", seed=99)

r_ok = sum(1 for r in router_results if r.status == "success")
b_ok = sum(1 for r in baseline_results if r.status == "success")
print(f"Generated {NUM_PROMPTS} prompts across {len(CATEGORIES)} categories")
print(f"  Router:   {r_ok}/{NUM_PROMPTS} successful")
print(f"  Baseline: {b_ok}/{NUM_PROMPTS} successful")

## 4. Compute Cost & Latency Metrics

The pricing table maps model names to per-million-token rates. `compute_metrics` aggregates cost, latency, and reliability.

In [ ]:
pricing = {
    "model_router": PricingConfig(input=0.50, output=1.50),
    "gpt-4o": PricingConfig(input=2.50, output=10.00),
}

category_map = {p.id: p.category for p in prompts if p.category}
metrics = compute_metrics(router_results, baseline_results, pricing, category_map)

# -- Cost summary --
rc, bc = metrics.model_router.cost, metrics.baseline.cost
print("=== Cost ===")
print(f"  Router:   ${rc.estimated_cost_usd:.4f}  ({rc.total_tokens:,} tokens)")
print(f"  Baseline: ${bc.estimated_cost_usd:.4f}  ({bc.total_tokens:,} tokens)")
print(f"  Savings:  {metrics.comparison.cost_savings_ratio:.0%}\n")

# -- Latency summary --
rl, bl = metrics.model_router.latency, metrics.baseline.latency
print("=== Latency (ms) ===")
print(f"  {'':>12} {'Router':>10} {'Baseline':>10}")
for stat in ["mean_ms", "median_ms", "p90_ms", "p95_ms", "p99_ms"]:
    label = stat.replace("_ms", "").replace("_", " ").title()
    print(f"  {label:>12} {getattr(rl, stat):>10.0f} {getattr(bl, stat):>10.0f}")
print(f"\n  Router is {abs(metrics.comparison.latency_diff_mean_ms):.0f}ms "
      f"{'faster' if metrics.comparison.latency_diff_mean_ms < 0 else 'slower'} on average")

# -- Reliability --
print(f"\n=== Reliability ===")
print(f"  Router:   {metrics.model_router.successful_requests}/{metrics.model_router.total_requests} success")
print(f"  Baseline: {metrics.baseline.successful_requests}/{metrics.baseline.total_requests} success")

## 5. Compute Quality Metrics (LLM-as-a-Judge)

The judge evaluates each response pair with **pairwise** (who wins?) and **absolute** (1–5 per dimension) scoring. Here we generate mock judge verdicts.

In [ ]:
def make_judge_results(prompts, seed=77):
    rng = random.Random(seed)
    results = []
    for p in prompts:
        if rng.random() < 0.03:
            results.append(JudgeResult(prompt_id=p.id, error="Mock judge timeout"))
            continue
        roll = rng.random()
        winner = "model_router" if roll < 0.55 else ("baseline" if roll < 0.80 else "tie")
        base_q = rng.gauss(3.8, 0.6)
        r_b = 0.4 if winner == "model_router" else (-0.2 if winner == "baseline" else 0.0)
        b_b = 0.4 if winner == "baseline" else (-0.2 if winner == "model_router" else 0.0)
        def _s(base, bonus):
            return max(1, min(5, round(base + bonus + rng.gauss(0, 0.3))))
        results.append(JudgeResult(
            prompt_id=p.id, pairwise_winner=winner,
            router_score=AbsoluteScore(
                accuracy=_s(base_q + r_b, rng.gauss(0, 0.2)),
                completeness=_s(base_q + r_b, rng.gauss(0, 0.2)),
                clarity=_s(base_q + r_b, rng.gauss(0, 0.2)),
                helpfulness=_s(base_q + r_b, rng.gauss(0, 0.2)),
            ),
            baseline_score=AbsoluteScore(
                accuracy=_s(base_q + b_b, rng.gauss(0, 0.2)),
                completeness=_s(base_q + b_b, rng.gauss(0, 0.2)),
                clarity=_s(base_q + b_b, rng.gauss(0, 0.2)),
                helpfulness=_s(base_q + b_b, rng.gauss(0, 0.2)),
            ),
            judge_model="gpt-4o", latency_ms=round(rng.gauss(2000, 500), 2),
        ))
    return results


judge_results = make_judge_results(prompts)
quality = compute_quality_metrics(
    judge_results=judge_results,
    category_map=category_map,
    router_cost_usd=metrics.model_router.cost.estimated_cost_usd,
    baseline_cost_usd=metrics.baseline.cost.estimated_cost_usd,
    router_mean_latency_ms=metrics.model_router.latency.mean_ms,
    baseline_mean_latency_ms=metrics.baseline.latency.mean_ms,
)
metrics.quality = quality

print("=== Pairwise Results ===")
print(f"  Router wins:   {quality.router_wins:>3}  ({quality.router_win_rate:.0%})")
print(f"  Baseline wins: {quality.baseline_wins:>3}  ({quality.baseline_win_rate:.0%})")
print(f"  Ties:          {quality.ties:>3}  ({quality.tie_rate:.0%})")
print(f"  Total judged:  {quality.total_judged}")

if quality.router_win_rate_ci:
    lo, hi = quality.router_win_rate_ci
    print(f"  95% CI:        [{lo:.0%}, {hi:.0%}]")

print(f"\n=== Absolute Scores (mean) ===")
print(f"  {'Dimension':<16} {'Router':>8} {'Baseline':>8}")
print(f"  {'-'*32}")
for dim in ["accuracy", "completeness", "clarity", "helpfulness"]:
    rs = quality.router_by_dimension.get(dim)
    bs = quality.baseline_by_dimension.get(dim)
    print(f"  {dim:<16} {rs.mean if rs else 0:>8.2f} {bs.mean if bs else 0:>8.2f}")
if quality.router_overall and quality.baseline_overall:
    print(f"  {'OVERALL':<16} {quality.router_overall.mean:>8.2f} {quality.baseline_overall.mean:>8.2f}")

print(f"\n=== Composite Scores ===")
print(f"  Value  (quality/cost):    Router {quality.router_value_score:>8.1f}  vs  Baseline {quality.baseline_value_score:>8.1f}")
print(f"  Efficiency (quality/lat): Router {quality.router_efficiency_score:>8.4f}  vs  Baseline {quality.baseline_efficiency_score:>8.4f}")

## 6. Generate Charts (Inline)

The evaluation produces 8 chart types. Here we generate them to a temp directory and display inline.

In [ ]:
import tempfile
from IPython.display import Image, display, HTML

# generate_all_charts uses the Agg backend — fine for saving to files
from src.charts import generate_all_charts

router_latencies = [r.latency_ms for r in router_results if r.status == "success"]
baseline_latencies = [r.latency_ms for r in baseline_results if r.status == "success"]

chart_dir = Path(tempfile.mkdtemp())
chart_files = generate_all_charts(
    metrics=metrics,
    router_latencies=router_latencies,
    baseline_latencies=baseline_latencies,
    output_dir=chart_dir,
    baseline_label="GPT-4o",
)

print(f"Generated {len(chart_files)} charts:\n")
for f in chart_files:
    print(f"  📊 {f}")
    display(Image(filename=str(chart_dir / f), width=700))
    print()

## 7. Generate the Full HTML Dashboard

This is the same self-contained dashboard that a real evaluation produces. It embeds all charts as base64 images — no external files needed.

In [ ]:
from src.dashboard import generate_dashboard

dashboard_file = generate_dashboard(
    metrics=metrics,
    eval_name="Notebook Walkthrough (Mock Data)",
    baseline_label="GPT-4o",
    chart_files=chart_files,
    output_dir=chart_dir,
)

dashboard_path = chart_dir / dashboard_file
print(f"Dashboard saved to: {dashboard_path}")
print(f"Open in browser for the full interactive experience.\n")

# Show a preview (first 2000 chars of the HTML)
html_content = dashboard_path.read_text(encoding="utf-8")
print(f"Dashboard size: {len(html_content):,} bytes ({len(html_content)//1024} KB)")
print(f"Self-contained: {'Yes' if 'data:image/png;base64' in html_content else 'No'}")

## 8. Per-Category Breakdown

Explore the data yourself — here's latency by category and quality win rates by category.

In [ ]:
# -- Latency by category --
print("=== Latency by Category (median ms) ===")
print(f"  {'Category':<24} {'Router':>10} {'Baseline':>10} {'Δ':>10}")
print(f"  {'-'*54}")
for cat in sorted(metrics.model_router.latency_by_category.keys()):
    rl = metrics.model_router.latency_by_category.get(cat)
    bl = metrics.baseline.latency_by_category.get(cat)
    if rl and bl:
        diff = rl.median_ms - bl.median_ms
        print(f"  {cat:<24} {rl.median_ms:>10.0f} {bl.median_ms:>10.0f} {diff:>+10.0f}")

# -- Win rate by category --
if quality.win_rate_by_category:
    print(f"\n=== Quality Win Rate by Category ===")
    print(f"  {'Category':<24} {'Router':>10} {'Baseline':>10} {'Tie':>10}")
    print(f"  {'-'*54}")
    for cat in sorted(quality.win_rate_by_category.keys()):
        wr = quality.win_rate_by_category[cat]
        print(f"  {cat:<24} {wr.get('model_router', 0):>9.0%} {wr.get('baseline', 0):>9.0%} {wr.get('tie', 0):>9.0%}")

## 9. Go Live — What to Change

To run a real evaluation, you only need to:

1. **Create `.env`** with your Azure credentials (see `.env.example`)
2. **Prepare a dataset** — JSONL, CSV, or a database connection string (or use the sample)
3. **Run from the CLI:**

```bash
# Quick test (5 prompts, no judge)
python scripts/run_eval.py --config configs/quick_test.yaml

# Full evaluation (all prompts, with LLM-as-a-judge)
python scripts/run_eval.py

# CSV or database source
python scripts/run_eval.py --dataset my_prompts.csv
python scripts/run_eval.py --dataset "sqlite:///prompts.db?table=prompts"

# Large scale with checkpoint/resume
python scripts/run_eval.py --config configs/large_scale.yaml --resume
```

Or programmatically (uncomment and fill in):

In [ ]:
# ── Uncomment below to run a live evaluation from this notebook ──

# from src.config import load_config
# from src.runner import run_evaluation
#
# config = load_config(project_root / "configs" / "quick_test.yaml")
# results = await run_evaluation(config)
#
# print(f"Done! Results saved to: {config.output_directory}")

## 10. Foundry Cloud Evaluation (Optional)

After running a local evaluation, you can submit the results to **Microsoft Foundry** for independent cloud-based grading. This uses the OpenAI Evals API with 5 graders:

| Grader | Type | What it measures |
|--------|------|-----------------|
| `quality_absolute_router` | `score_model` | Router response quality (1–5, pass ≥ 3) |
| `quality_absolute_baseline` | `score_model` | Baseline response quality (1–5, pass ≥ 3) |
| `quality_pairwise` | `score_model` | Head-to-head comparison (1–5, pass ≥ 3) |
| `mr_cost_comparison` | `python` | Cost savings ratio (pass ≥ 0.5) |
| `mr_latency_comparison` | `python` | Latency improvement ratio (pass ≥ 0.5) |

**Prerequisites:** `pip install -e ".[foundry]"`, `az login`, and `AZURE_AI_PROJECT_ENDPOINT` in `.env`.

In [ ]:
# ── Submit local results to Foundry for cloud grading ──
# Requires: pip install -e ".[foundry]", az login, AZURE_AI_PROJECT_ENDPOINT in .env

import subprocess, json
from pathlib import Path

input_dir = project_root / "results" / "full-eval"
foundry_dir = project_root / "results" / "foundry-eval"

if not (input_dir / "raw_results.jsonl").exists():
    print("⚠ No local eval results found. Run a live evaluation first (Cell 11).")
else:
    # Run Foundry eval via CLI (same as: python scripts/run_foundry_eval.py --input-dir results/full-eval)
    print("Submitting to Foundry... (this takes 1–3 minutes)")
    result = subprocess.run(
        ["python", "scripts/run_foundry_eval.py", "--input-dir", str(input_dir)],
        capture_output=True, text=True, cwd=str(project_root),
    )
    if result.returncode == 0:
        print(result.stdout[-500:])  # Last 500 chars of output
        # Show grader summary
        foundry_results = json.loads((foundry_dir / "results.json").read_text())
        print("\n── Grader Summary ──")
        for grader, stats in foundry_results.get("grader_summary", {}).items():
            print(f"  {grader:<30s} mean={stats['mean']:.2f}  pass_rate={stats['pass_rate']:.0f}%  n={stats['count']}")
    else:
        print(f"✗ Foundry eval failed:\n{result.stderr[-500:]}")

## 11. Cross-Validate: Local vs Foundry

Compare local evaluation scores against Foundry cloud scores to confirm they agree. Both pipelines independently grade the same prompt/response pairs — strong correlation validates your results.

In [ ]:
# ── Cross-validate local eval vs Foundry cloud eval ──

import json
from pathlib import Path

foundry_path = project_root / "results" / "foundry-eval" / "results.json"
local_path = project_root / "results" / "full-eval" / "results.json"

if not foundry_path.exists() or not local_path.exists():
    print("⚠ Need both results/full-eval/results.json and results/foundry-eval/results.json")
    print("  Run local eval (Cell 11) then Foundry eval (Cell 13) first.")
else:
    foundry = json.loads(foundry_path.read_text())
    local = json.loads(local_path.read_text())
    gs = foundry.get("grader_summary", {})

    lr = local["quality"]["absolute_scores"]["router_overall"]
    lb = local["quality"]["absolute_scores"]["baseline_overall"]
    fr = gs.get("quality_absolute_router", {}).get("mean", 0)
    fb = gs.get("quality_absolute_baseline", {}).get("mean", 0)
    lc = local["comparison"]["cost_savings_ratio"]
    fc = gs.get("mr_cost_comparison", {}).get("mean", 0)
    ll = local["comparison"]["latency_diff_mean_ms"]
    fl = gs.get("mr_latency_comparison", {}).get("mean", 0)
    flp = gs.get("mr_latency_comparison", {}).get("pass_rate", 0)

    print("=" * 65)
    print("  CROSS-VALIDATION: Local Eval vs Foundry Cloud Eval")
    print("=" * 65)

    print(f"\n  {'Metric':<25} {'Local':<15} {'Foundry':<15} {'Agree?'}")
    print("  " + "-" * 60)

    # Quality
    q_agree = (lr > lb) == (fr > fb)
    print(f"  {'Router quality':<25} {lr:<15.2f} {fr:<15.2f} {'✓' if abs(fr-lr)<1.0 else '✗'}")
    print(f"  {'Baseline quality':<25} {lb:<15.2f} {fb:<15.2f} {'✓' if abs(fb-lb)<1.0 else '✗'}")
    print(f"  {'Router > Baseline?':<25} {'Yes':<15} {'Yes' if fr>fb else 'No':<15} {'✓' if q_agree else '✗'}")

    # Cost
    c_agree = abs(fc - lc) < 0.05
    print(f"  {'Cost savings':<25} {lc:<15.1%} {fc:<15.1%} {'✓' if c_agree else '✗'}")

    # Latency
    print(f"  {'Latency':<25} {'+' + str(round(ll)) + 'ms':<15} {flp:.0f}% pass     {'✓' if fl < 0.6 else '✗'}")

    # Per-item scores table
    per_item = foundry.get("per_item_scores", [])
    if per_item:
        print(f"\n  {'Prompt':<12} {'RtrQ':>5} {'BaseQ':>6} {'Pair':>5} {'Cost':>7} {'Latency':>8}")
        print("  " + "-" * 50)
        for item in per_item:
            pid = item["prompt_id"][-3:]
            s = item["scores"]
            rq = s.get("quality_absolute_router", {}).get("score", 0)
            bq = s.get("quality_absolute_baseline", {}).get("score", 0)
            pw = s.get("quality_pairwise", {}).get("score", 0)
            co = s.get("mr_cost_comparison", {}).get("score", 0)
            la = s.get("mr_latency_comparison", {}).get("score", 0)
            lm = " *" if not s.get("mr_latency_comparison", {}).get("passed", True) else ""
            bm = " *" if not s.get("quality_absolute_baseline", {}).get("passed", True) else ""
            print(f"  {pid:<12} {rq:>5.0f} {bq:>5.0f}{bm:2s} {pw:>5.0f} {co:>7.3f} {la:>7.3f}{lm}")
        print("  * = failed threshold")

    print(f"\n  VERDICT: {'CORRELATE WELL ✓' if q_agree and c_agree else 'DIVERGENCE DETECTED — investigate'}")